In [0]:
from pyspark.sql import Window, functions as F

# DBTITLE 1,Configuración de la capa
bronze = "workspace.bronze.adidas_us_sales"
silver = "workspace.silver.adidas_us_sales"

checkpoint = "/Volumes/workspace/silver/checkpoint"
checkpoint = "/Volumes/workspace/silver/checkpoint/reinicio_cdf"
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.silver.checkpoint")


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver} (
      id_venta BIGINT, tienda STRING, fecha_factura DATE, region STRING,
      estado STRING, ciudad STRING, producto STRING, precio_unitario INT,
      unidades_vendidas INT, total_ventas INT, ganancia_operativa INT, canal_venta STRING
    ) USING DELTA TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

In [0]:
def ultimos_cambios(lote, clave):
    eventos = lote.filter(F.col("_change_type") != "update_preimage")
    orden = Window.partitionBy(clave).orderBy(F.col("_commit_version").desc())
    return (
        eventos.withColumn("_orden", F.row_number().over(orden))
        .filter("_orden = 1")
        .drop("_orden", "_commit_version", "_commit_timestamp")
    )

In [0]:
def aplicar_cambios(datos, destino, clave, columnas):
    sesion = datos.sparkSession
    datos.createOrReplaceTempView("cambios_pipeline")
    asignaciones = ", ".join(f"t.{c} = s.{c}" for c in columnas if c != clave)
    diferencias = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in columnas if c != clave)
    nombres = ", ".join(columnas)
    valores = ", ".join(f"s.{c}" for c in columnas)
    sesion.sql(f"""
        MERGE INTO {destino} t USING cambios_pipeline s ON t.{clave} = s.{clave}
        WHEN MATCHED AND s._borrar THEN DELETE
        WHEN MATCHED AND NOT s._borrar AND ({diferencias}) THEN UPDATE SET {asignaciones}
        WHEN NOT MATCHED AND NOT s._borrar THEN INSERT ({nombres}) VALUES ({valores})
    """)

In [0]:
def transformar_silver(lote, _):
    if lote.isEmpty():
        return

    if lote.filter("sales_id IS NULL").limit(1).count():
        raise ValueError("Bronze contiene una fila sin sales_id.")
    eventos = lote.filter("_change_type <> 'update_preimage'").dropDuplicates()
    if eventos.groupBy("sales_id", "_commit_version").count().filter("count > 1").limit(1).count():
        raise ValueError("Hay versiones contradictorias de un sales_id en el mismo commit.")

    limpios = ultimos_cambios(eventos, "sales_id").selectExpr(
        "sales_id AS id_venta", "upper(trim(retailer)) AS tienda",
        "invoice_date AS fecha_factura", "upper(trim(region)) AS region",
        "upper(trim(state)) AS estado", "upper(trim(city)) AS ciudad",
        "upper(trim(product)) AS producto", "price_per_unit AS precio_unitario",
        "units_sold AS unidades_vendidas", "total_sales AS total_ventas",
        "operating_profit AS ganancia_operativa", "upper(trim(sales_method)) AS canal_venta",
        "_change_type",
    )
    columnas = [c for c in limpios.columns if c != "_change_type"]
    obligatorios = " AND ".join(f"{columna} IS NOT NULL" for columna in columnas)
    textos = " AND ".join(
        f"{columna} <> ''" for columna in ("tienda", "region", "estado", "ciudad", "producto", "canal_venta")
    )
    cambios = limpios.withColumn(
        "_borrar", F.expr(f"_change_type = 'delete' OR NOT ({obligatorios} AND {textos})")
    )
    aplicar_cambios(cambios, silver, "id_venta", columnas)


In [0]:
flujo = (
    spark.readStream.option("readChangeFeed", "true").table(bronze)
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(transformar_silver)
    .start()
)
flujo.awaitTermination()
